In [1]:
# Cell 1: Imports and Setup
import json
import re
from pathlib import Path
from typing import List, Dict

print("✓ Imports loaded")

✓ Imports loaded


In [ ]:
# # Cell 2: Updated Violation Finder with Better Patterns
# def find_lowercase_suffix_violations(code: str) -> List[Dict]:
#     """
#     Find literals with lowercase suffixes (u, l, f, ul, ll, etc.)
    
#     UPDATED: More comprehensive pattern matching
#     """
#     violations = []
    
#     # More comprehensive patterns
#     patterns = [
#         # Hexadecimal with lowercase suffixes
#         (r'\b0x[0-9a-fA-F]+u\b', 'hex_u'),
#         (r'\b0x[0-9a-fA-F]+ul\b', 'hex_ul'),
#         (r'\b0x[0-9a-fA-F]+ull\b', 'hex_ull'),
#         (r'\b0x[0-9a-fA-F]+l\b', 'hex_l'),
#         (r'\b0x[0-9a-fA-F]+ll\b', 'hex_ll'),
#         (r'\b0x[0-9a-fA-F]+lu\b', 'hex_lu'),          # Alternative order
#         (r'\b0x[0-9a-fA-F]+llu\b', 'hex_llu'),        # Alternative order
        
#         # Decimal with lowercase suffixes
#         (r'\b\d+u\b', 'dec_u'),
#         (r'\b\d+ul\b', 'dec_ul'),
#         (r'\b\d+ull\b', 'dec_ull'),
#         (r'\b\d+l\b', 'dec_l'),
#         (r'\b\d+ll\b', 'dec_ll'),
#         (r'\b\d+lu\b', 'dec_lu'),
#         (r'\b\d+llu\b', 'dec_llu'),
        
#         # Floats with lowercase suffixes
#         (r'\b\d+\.\d+f\b', 'float_f'),
#         (r'\b\d+\.\d*f\b', 'float_f_alt'),             # e.g., 1.f
#         (r'\b\d*\.\d+f\b', 'float_f_alt2'),            # e.g., .5f
#         (r'\b\d+f\b', 'int_f'),
#         (r'\b\d+\.\d+l\b', 'float_l'),
#         (r'\b\d+\.?\d*e[+-]?\d+f\b', 'scientific_f'),  # Scientific notation
        
#         # Octal with lowercase suffixes (starts with 0)
#         (r'\b0[0-7]+u\b', 'oct_u'),
#         (r'\b0[0-7]+ul\b', 'oct_ul'),
#         (r'\b0[0-7]+l\b', 'oct_l'),
#     ]
    
#     for pattern, viol_type in patterns:
#         for match in re.finditer(pattern, code, re.IGNORECASE):
#             literal = match.group(0)
            
#             # Check if it actually has lowercase suffix
#             # (pattern is case-insensitive, so we need to verify)
#             if literal == literal.upper():
#                 continue  # Skip if already uppercase
            
#             # Check if suffix is lowercase
#             has_lowercase_suffix = False
#             for suffix_char in ['u', 'l', 'f']:
#                 if suffix_char in literal.lower() and suffix_char in literal:
#                     has_lowercase_suffix = True
#                     break
            
#             if not has_lowercase_suffix:
#                 continue
            
#             line_num = code[:match.start()].count('\n') + 1
            
#             violations.append({
#                 'literal': literal,
#                 'fixed': literal.upper(),
#                 'type': viol_type,
#                 'line': line_num
#             })
    
#     return violations

# print("✓ Updated violation finder function defined")

✓ Updated violation finder function defined


In [2]:
def find_multiple_declarations_violations(code: str) -> List[Dict]:
    """
    Find multiple variable declarations on the same line (MISRA C++ Rule 8-0-1)
    
    Detects patterns like:
    - int a, b, c;
    - int *p, q;
    - MyClass obj1, obj2;
    - int a = 1, b = 2;
    
    Does NOT flag:
    - Function parameters: void func(int a, int b)
    - Template parameters: template<class T, class U>
    - For loop initializers: for(int i = 0, j = 0; ...)
    """
    violations = []
    
    # Split code into lines
    lines = code.split('\n')
    
    for line_num, line in enumerate(lines, 1):
        # Skip empty lines and comments
        stripped = line.strip()
        if not stripped or stripped.startswith('//') or stripped.startswith('/*') or stripped.startswith('*'):
            continue
        
        # Remove string literals to avoid false positives
        temp_line = re.sub(r'"([^"\\]|\\.)*"', '""', line)
        temp_line = re.sub(r"'([^'\\]|\\.)*'", "''", temp_line)
        
        # Remove inline comments
        temp_line = re.sub(r'//.*$', '', temp_line)
        
        # Skip preprocessor directives
        if '#' in temp_line:
            continue
        
        # Skip function parameters (content within function declaration parentheses)
        # Pattern: word followed by ( ... ) with commas inside
        if re.search(r'\w+\s*\([^)]*,', temp_line):
            # Check if this looks like a function declaration/definition
            # by checking if there's a parameter list pattern
            if re.search(r'\w+\s*\([^)]*\w+\s+\w+[^)]*,', temp_line):
                continue
        
        # Skip template declarations
        if re.search(r'template\s*<', temp_line):
            continue
        
        # Skip for loop initializers
        if re.search(r'\bfor\s*\(', temp_line):
            continue
        
        # Main pattern: Look for variable declarations with commas
        # Pattern explanation:
        # - Type name (possibly with const, static, etc.)
        # - Variable name (possibly with pointer/reference *)
        # - Comma
        # - Another variable name
        
        # Pattern for basic declarations: type var1, var2;
        patterns = [
            # Standard types: int a, b;
            r'\b(const\s+|static\s+|volatile\s+|extern\s+|mutable\s+)*(unsigned\s+|signed\s+)?(char|short|int|long|float|double|bool|void|size_t|uint8_t|uint16_t|uint32_t|uint64_t|int8_t|int16_t|int32_t|int64_t)\s+(\*|\&)*\s*\w+\s*(\[[^\]]*\])?\s*(=[^,;]+)?\s*,',
            
            # Custom types/classes: MyClass obj1, obj2;
            r'\b(const\s+|static\s+|volatile\s+)?\b[A-Z]\w*\s+(\*|\&)*\s*\w+\s*(\[[^\]]*\])?\s*(=[^,;]+)?\s*,',
            
            # Auto type: auto a = 1, b = 2;
            r'\bauto\s+(\*|\&)*\s*\w+\s*(=[^,;]+)?\s*,',
            
            # With namespace: std::string s1, s2;
            r'\b\w+::\w+\s+(\*|\&)*\s*\w+\s*(\[[^\]]*\])?\s*(=[^,;]+)?\s*,',
        ]
        
        for pattern in patterns:
            matches = list(re.finditer(pattern, temp_line))
            
            if matches:
                # Extract the full declaration statement
                # Try to find the semicolon to get the complete statement
                stmt_match = re.search(r'[^;{]*;', temp_line)
                statement = stmt_match.group(0) if stmt_match else temp_line
                
                # Count the number of variable declarations (commas + 1)
                # But be careful with initializers that contain commas
                # Simple heuristic: count commas outside of parentheses and braces
                
                # Remove content within parentheses and braces for counting
                temp_stmt = statement
                temp_stmt = re.sub(r'\([^)]*\)', '', temp_stmt)
                temp_stmt = re.sub(r'\{[^}]*\}', '', temp_stmt)
                
                comma_count = temp_stmt.count(',')
                
                if comma_count > 0:
                    violations.append({
                        'line': line_num,
                        'statement': statement.strip(),
                        'original_line': line.strip(),
                        'num_variables': comma_count + 1,
                        'type': 'multiple_declarations'
                    })
                    break  # Only report once per line
    
    return violations

print("✓ MISRA C++ Rule 8-0-1 violation finder function defined")

✓ MISRA C++ Rule 8-0-1 violation finder function defined


In [5]:
# Cell 4: Universal Scanner - Handles both JSON formats
def scan_primevul_dataset(json_file: str) -> List[Dict]:
    """
    Scan PrimeVul dataset and extract entries with MISRA 2-13-4 violations
    Handles both JSON array format and newline-delimited JSON
    
    Args:
        json_file: Path to PrimeVul JSON file
        
    Returns:
        List of examples with violations
    """
    examples = []
    
    print(f"Scanning {json_file} for MISRA 2-13-4 violations...")
    print("Rule: Literal suffixes shall be upper case\n")
    
    # First, try to load as complete JSON array
    try:
        with open(json_file, 'r', encoding='utf-8', errors='ignore') as f:
            data = json.load(f)
            
        print(f"Loaded JSON array with {len(data)} entries")
        
        for idx, entry in enumerate(data):
            # Get the function code
            code = entry.get('func', '')
            if not code:
                continue
            
            # Find violations
            violations = find_multiple_declarations_violations(code)
            
            if violations:
                examples.append({
                    'idx': entry.get('idx'),
                    'project': entry.get('project'),
                    'file_name': entry.get('file_name'),
                    'cve': entry.get('cve'),
                    "cve_desc": entry.get('cve_desc'),
                    'commit_url': entry.get('commit_url'),
                    'violations': violations,
                    'code': code
                })
                
                if len(examples) % 10 == 0:
                    print(f"  Found {len(examples)} examples so far...")
            
            if (idx + 1) % 500 == 0:
                print(f"  Processed {idx + 1} entries...")
    
    except json.JSONDecodeError:
        # If that fails, try newline-delimited JSON
        print("JSON array format failed, trying newline-delimited format...")
        
        with open(json_file, 'r', encoding='utf-8', errors='ignore') as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                
                if not line or line in ['[', ']']:
                    continue
                
                # Remove trailing comma
                if line.endswith(','):
                    line = line[:-1]
                
                try:
                    entry = json.loads(line)
                    
                    code = entry.get('func', '')
                    if not code:
                        continue
                    
                    violations = find_multiple_declarations_violations(code)
                    
                    if violations:
                        examples.append({
                            'idx': entry.get('idx'),
                            'project': entry.get('project'),
                            'file_name': entry.get('file_name'),
                            'cve': entry.get('cve'),
                            'commit_url': entry.get('commit_url'),
                            'violations': violations,
                            'code': code
                        })
                        
                        if len(examples) % 10 == 0:
                            print(f"  Found {len(examples)} examples so far...")
                    
                    if line_num % 500 == 0:
                        print(f"  Processed {line_num} lines...")
                            
                except json.JSONDecodeError:
                    continue
    
    print(f"\n✓ Scan complete!")
    print(f"✓ Found {len(examples)} examples with violations")
    
    return examples

print("✓ Scanner function defined")

✓ Scanner function defined


In [16]:
# Cell 5: Run the Scanner
# UPDATE THIS PATH to your PrimeVul dataset file
dataset_file = 'primevul_test_paired.jsonl'  # Change this to your file path

examples = scan_primevul_dataset(dataset_file)

Scanning primevul_test_paired.jsonl for MISRA 2-13-4 violations...
Rule: Literal suffixes shall be upper case

JSON array format failed, trying newline-delimited format...
  Found 10 examples so far...
  Found 20 examples so far...
  Found 30 examples so far...
  Found 40 examples so far...
  Found 50 examples so far...
  Found 60 examples so far...
  Found 70 examples so far...
  Found 80 examples so far...
  Found 90 examples so far...
  Found 100 examples so far...
  Found 110 examples so far...
  Found 120 examples so far...
  Found 130 examples so far...
  Found 140 examples so far...
  Found 150 examples so far...
  Found 160 examples so far...
  Found 170 examples so far...
  Found 180 examples so far...
  Found 190 examples so far...
  Processed 500 lines...
  Found 200 examples so far...
  Found 210 examples so far...
  Found 220 examples so far...
  Found 230 examples so far...
  Found 240 examples so far...
  Found 250 examples so far...
  Found 260 examples so far...
  Foun

In [17]:
# Cell 6: Display Summary Statistics
print("="*70)
print("SUMMARY STATISTICS")
print("="*70)

# Count violation types
violation_types = {}
for ex in examples:
    for v in ex['violations']:
        vtype = v['type']
        violation_types[vtype] = violation_types.get(vtype, 0) + 1

print("\nViolation Types Distribution:")
for vtype, count in sorted(violation_types.items(), key=lambda x: x[1], reverse=True):
    print(f"  {vtype:15s}: {count:4d}")

print(f"\nTotal Examples: {len(examples)}")
print(f"Total Violations: {sum(len(ex['violations']) for ex in examples)}")
# print(f"Avg Violations per Example: {sum(len(ex['violations']) for ex in examples) / len(examples):.2f}")

SUMMARY STATISTICS

Violation Types Distribution:
  multiple_declarations:  738

Total Examples: 354
Total Violations: 738


In [12]:
examples[:2]

[{'idx': 195042,
  'project': 'tensorflow',
  'file_name': 'fully_connected.cc',
  'cve': 'CVE-2022-23561',
  'commit_url': 'https://github.com/tensorflow/tensorflow/commit/6c0b2b70eeee588591680f5b7d5d38175fd7cdf6',
  'violations': [{'line': 5,
    'statement': 'float output_activation_min, output_activation_max;',
    'original_line': 'float output_activation_min, output_activation_max;',
    'num_variables': 2,
    'type': 'multiple_declarations'}],
  'code': 'TfLiteStatus EvalFloat(TfLiteContext* context, TfLiteNode* node,\n                       TfLiteFullyConnectedParams* params, OpData* data,\n                       const TfLiteTensor* input, const TfLiteTensor* filter,\n                       const TfLiteTensor* bias, TfLiteTensor* output) {\n  float output_activation_min, output_activation_max;\n  CalculateActivationRange(params->activation, &output_activation_min,\n                           &output_activation_max);\n  if (kernel_type == kReference) {\n    FullyConnectedParams

In [18]:
# valid_examples = examples.copy()
test_examples = examples.copy()


In [21]:
examples = test_examples + valid_examples

In [22]:
len(examples)

1238

In [23]:
# Cell 10: Save Examples to JSON (Optional)
import json

output_file = 'misra_8_0_1_pairs.json'

with open(output_file, 'w') as f:
    json.dump(examples, f, indent=2)

print(f"✓ Saved {len(examples)} examples to: {output_file}")

✓ Saved 1238 examples to: misra_8_0_1_pairs.json


In [146]:
examples[0]

{'idx': 9,
 'project': 'ghostscript',
 'file_name': 'None',
 'cve': 'CVE-2018-1000037',
 'commit_url': 'http://git.ghostscript.com/?p=mupdf.git;a=commitdiff;h=b2e7d38e845c7d4922d05e6e41f3a2dc1bc1b14a;hp=f51836b9732c38d945b87fda0770009a77ba680c',
 'violations': [{'literal': '1.0f',
   'fixed': '1.0F',
   'type': 'float_f',
   'line': 92},
  {'literal': '1.0f', 'fixed': '1.0F', 'type': 'float_f_alt', 'line': 92},
  {'literal': '1.0f', 'fixed': '1.0F', 'type': 'float_f_alt2', 'line': 92},
  {'literal': '0f', 'fixed': '0F', 'type': 'int_f', 'line': 92}],
 'code': " pdf_show_image(fz_context *ctx, pdf_run_processor *pr, fz_image *image)\n {\n        pdf_gstate *gstate = pr->gstate + pr->gtop;\n        fz_matrix image_ctm;\n        fz_rect bbox;\n       softmask_save softmask = { NULL };\n \n        if (pr->super.hidden)\n                return;\n\t\t\tbreak;\n\t\tcase PDF_MAT_SHADE:\n\t\t\tif (gstate->fill.shade)\n\t\t\t{\n\t\t\t\tfz_clip_image_mask(ctx, pr->dev, image, &image_ctm, &bbox);\

In [147]:
examples

[{'idx': 9,
  'project': 'ghostscript',
  'file_name': 'None',
  'cve': 'CVE-2018-1000037',
  'commit_url': 'http://git.ghostscript.com/?p=mupdf.git;a=commitdiff;h=b2e7d38e845c7d4922d05e6e41f3a2dc1bc1b14a;hp=f51836b9732c38d945b87fda0770009a77ba680c',
  'violations': [{'literal': '1.0f',
    'fixed': '1.0F',
    'type': 'float_f',
    'line': 92},
   {'literal': '1.0f', 'fixed': '1.0F', 'type': 'float_f_alt', 'line': 92},
   {'literal': '1.0f', 'fixed': '1.0F', 'type': 'float_f_alt2', 'line': 92},
   {'literal': '0f', 'fixed': '0F', 'type': 'int_f', 'line': 92}],
  'code': " pdf_show_image(fz_context *ctx, pdf_run_processor *pr, fz_image *image)\n {\n        pdf_gstate *gstate = pr->gstate + pr->gtop;\n        fz_matrix image_ctm;\n        fz_rect bbox;\n       softmask_save softmask = { NULL };\n \n        if (pr->super.hidden)\n                return;\n\t\t\tbreak;\n\t\tcase PDF_MAT_SHADE:\n\t\t\tif (gstate->fill.shade)\n\t\t\t{\n\t\t\t\tfz_clip_image_mask(ctx, pr->dev, image, &image_